# Workspace Registry Migration

Bulk-migrates MLflow **workspace registry** models and experiments from a source workspace into the current (target) workspace under `/Shared`.

### Quick-start
1. Configure **credentials** — either a PAT token or a Service Principal (client_id / client_secret stored in Databricks Secrets).
2. Review **migration options** — prefixes, batch size, artifact handling, and optional allowlists.
3. Run **discovery** to preview what will be migrated.
4. Execute **bulk migration** to copy experiments, runs, and registered model versions to the target workspace registry.

## Prerequisites

### Network Connectivity
* The **target workspace** (where this notebook runs) must have network connectivity to the **source workspace** REST API.
* If either workspace uses a Private Link / VNet-injected deployment, ensure:
  * The target cluster/serverless compute can reach the source workspace's control-plane endpoint (port 443).
  * NSG / firewall rules allow HTTPS egress from the target to the source workspace URL.
  * If a private DNS zone is in use, the source workspace FQDN resolves correctly from the target network.
* If workspaces are in **different regions or subscriptions**, verify there is VNet peering, transit gateway, or public egress available.

### Authentication (choose one)
| Method | What to provide |
| --- | --- |
| **PAT** | Source workspace host + personal access token (stored in a Databricks secret scope recommended). |
| **Service Principal (OAuth M2M)** | Source workspace host + `client_id` + `client_secret` stored in a Databricks secret scope. The SP must be registered in the source workspace with workspace-level access. |

### Required Permissions on the Source Workspace
* Read access to workspace MLflow Tracking (experiments and runs).
* Read access to workspace Model Registry (registered models and versions).
* Artifact download permissions (if `include_run_artifacts=True`).

### Required Permissions on the Target Workspace (current)
* Write to `/Shared` experiments directory.
* Create registered models and model versions in the workspace registry.

### Secret Scope Setup (recommended)
```
# One-time setup — store credentials in Databricks Secrets
databricks secrets create-scope migration-scope
databricks secrets put-secret migration-scope source-host
databricks secrets put-secret migration-scope source-token          # if using PAT
databricks secrets put-secret migration-scope sp-client-id          # if using SP
databricks secrets put-secret migration-scope sp-client-secret      # if using SP
```

In [0]:
import importlib
import os
import sys
from dataclasses import asdict

# aDerive project root from notebook path (works for any user)
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root = "/Workspace" + str(os.path.dirname(_notebook_path))
if project_root not in sys.path:
    sys.path.append(project_root)

importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "workspace_registry_migrator.framework" or module_name.startswith("workspace_registry_migrator."):
        sys.modules.pop(module_name, None)

import workspace_registry_migrator.framework as framework
framework = importlib.reload(framework)

MigrationOptions = framework.MigrationOptions
SourceWorkspaceCredentials = framework.SourceWorkspaceCredentials
build_migrator = framework.build_migrator


In [0]:
from databricks.sdk import WorkspaceClient

# --------------------------------------------------------------------------
# AUTH MODE: Choose "pat" or "service_principal"
# --------------------------------------------------------------------------
AUTH_MODE = "pat"  # <-- Change to "service_principal" to use SP credentials

# Source workspace credentials
# Defaults to current workspace context; replace with remote workspace values for real migrations.
_ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
SOURCE_HOST = _ctx.apiUrl().get()  # Replace with remote workspace URL for real migrations
SOURCE_TOKEN = _ctx.apiToken().get()  # Replace with remote workspace PAT for real migrations
SP_CLIENT_ID = ""  # Service Principal client ID (if using SP)
SP_CLIENT_SECRET = ""  # Service Principal client secret (if using SP)

# --------------------------------------------------------------------------
current_workspace = WorkspaceClient()

if AUTH_MODE == "service_principal":
    credentials = SourceWorkspaceCredentials(
        host=SOURCE_HOST,
        token=None,
        client_id=SP_CLIENT_ID,
        client_secret=SP_CLIENT_SECRET,
        tracking_uri="databricks",
        registry_uri="databricks",
    )
else:
    credentials = SourceWorkspaceCredentials(
        host=SOURCE_HOST,
        token=SOURCE_TOKEN,
        tracking_uri="databricks",
        registry_uri="databricks",
    )

target_host = current_workspace.config.host

if credentials.host.rstrip("/") == target_host.rstrip("/"):
    print("\u26a0\ufe0f  WARNING: Source and target are the SAME workspace. "
          "Replace SOURCE_HOST/SOURCE_TOKEN with the remote workspace for a real migration.")

{
    "auth_mode": AUTH_MODE,
    "source_host (migrate FROM)": credentials.host,
    "target_host (migrate TO)": target_host,
    "auth_type": credentials.auth_type(),
}


In [0]:
import os
import warnings

os.environ.update(
    {
        "TQDM_DISABLE": "1",
        "MLFLOW_ENABLE_TQDM": "false",
        "MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR": "false",
    }
)
warnings.filterwarnings("ignore")

options = MigrationOptions(
    shared_experiment_root="/Shared/mlflow-workspace-migration",
    model_name_prefix="testcopyv8_",
    experiment_name_prefix="testv8_",
    batch_size=8,
    max_workers=12,
    download_artifacts=True,
    migrate_experiments=True,
    migrate_registered_models=True,
    include_run_artifacts=True,
    include_deleted_runs=True,
    max_runs_per_experiment=20,
    max_model_versions_per_model=3,
    create_missing_experiments=True,
    skip_existing_model_versions=True,
    extra_model_names=['airbnb-rf-model_5dbfdc', 'alex_airbnb', 'covid_forecast_tj'],
    extra_experiment_ids=[],
)

asdict(options)


In [0]:
import warnings

warnings.filterwarnings("ignore")

migrator = build_migrator(
    source_host=credentials.host,
    source_token=credentials.token,
    source_client_id=credentials.client_id,
    source_client_secret=credentials.client_secret,
    tracking_uri=credentials.tracking_uri,
    registry_uri=credentials.registry_uri,
    **asdict(options),
)

inventory = migrator.discover()

summary = {
    "registered_models": len(inventory.registered_models),
    "experiments": len(inventory.experiments),
    "runs": sum(len(runs) for runs in inventory.runs_by_experiment_id.values()),
    "model_versions": sum(len(versions) for versions in inventory.model_versions_by_name.values()),
}
summary


In [0]:
migrator = build_migrator(
    source_host=credentials.host,
    source_token=credentials.token,
    source_client_id=credentials.client_id,
    source_client_secret=credentials.client_secret,
    tracking_uri=credentials.tracking_uri,
    registry_uri=credentials.registry_uri,
    **asdict(options),
)
migrator.target_workspace.workspace.mkdirs(options.shared_experiment_root)
asdict(migrator.migrate_all())


In [0]:
import pandas as pd
from mlflow import MlflowClient

client = MlflowClient(tracking_uri="databricks", registry_uri="databricks")
prefix = options.model_name_prefix

# Collect detailed report rows
report_rows = []

# Get all migrated models
for source_model_name in [m.name for m in inventory.registered_models]:
    target_model_name = f"{prefix}{source_model_name}"
    escaped_src = source_model_name.replace("'", "''")
    escaped_tgt = target_model_name.replace("'", "''")
    
    source_versions = list(client.search_model_versions(filter_string=f"name='{escaped_src}'"))
    target_versions = list(client.search_model_versions(filter_string=f"name='{escaped_tgt}'"))
    
    # Map target versions by source_model_version tag
    target_by_source_v = {
        (tv.tags or {}).get("source_model_version"): tv
        for tv in target_versions
    }
    
    for sv in source_versions:
        tv = target_by_source_v.get(str(sv.version))
        
        # Source run check
        src_run_ok = False
        src_params = src_metrics = src_artifacts = 0
        try:
            src_run = client.get_run(sv.run_id)
            src_run_ok = True
            src_params = len(src_run.data.params)
            src_metrics = len(src_run.data.metrics)
            try:
                src_artifacts = len(client.list_artifacts(run_id=sv.run_id, path="model"))
            except:
                pass
        except:
            pass
        
        # Target comparison
        tgt_params = tgt_metrics = tgt_artifacts = 0
        params_match = metrics_match = artifacts_match = False
        if tv and tv.run_id:
            try:
                tgt_run = client.get_run(tv.run_id)
                tgt_params = len(tgt_run.data.params)
                tgt_metrics = len(tgt_run.data.metrics)
                params_match = (tgt_params == src_params)
                metrics_match = (tgt_metrics == src_metrics)
                try:
                    tgt_artifacts = len(client.list_artifacts(run_id=tv.run_id, path="model"))
                    artifacts_match = (tgt_artifacts == src_artifacts)
                except:
                    pass
            except:
                pass
        
        # Determine status
        if tv:
            status = "✅ MIGRATED"
        elif not src_run_ok:
            status = "❌ RUN_DELETED"
        elif src_artifacts == 0:
            status = "❌ NO_ARTIFACTS"
        else:
            status = "⚠️ FAILED"
        
        report_rows.append({
            "source_model": source_model_name,
            "version": sv.version,
            "target_model": target_model_name,
            "status": status,
            "source_run_accessible": src_run_ok,
            "source_params": src_params,
            "target_params": tgt_params if tv else None,
            "params_match": params_match if tv else None,
            "source_metrics": src_metrics,
            "target_metrics": tgt_metrics if tv else None,
            "metrics_match": metrics_match if tv else None,
            "source_model_files": src_artifacts,
            "target_model_files": tgt_artifacts if tv else None,
            "artifacts_match": artifacts_match if tv else None,
        })

report_df = pd.DataFrame(report_rows)

# Summary counts
total = len(report_df)
migrated = len(report_df[report_df["status"] == "✅ MIGRATED"])
failed = len(report_df[report_df["status"] == "⚠️ FAILED"])
deleted = len(report_df[report_df["status"] == "❌ RUN_DELETED"])
no_arts = len(report_df[report_df["status"] == "❌ NO_ARTIFACTS"])

print(f"MIGRATION REPORT: {migrated}/{total} versions migrated successfully")
print(f"  ✅ Migrated: {migrated}  |  ⚠️ Failed: {failed}  |  ❌ Run deleted: {deleted}  |  ❌ No artifacts: {no_arts}")
print()

# Show skipped versions from migration summary (if available)
try:
    result = asdict(migrator.migrate_all.__wrapped__) if hasattr(migrator, '_skipped_versions') else None
except:
    pass
if hasattr(migrator, '_skipped_versions') and migrator._skipped_versions:
    print(f"\nSKIPPED VERSIONS DETAIL ({len(migrator._skipped_versions)} failures):")
    skipped_df = pd.DataFrame(migrator._skipped_versions)
    display(skipped_df)
    print()

display(report_df)